# Colab 04 — Evaluación en el test sellado y análisis finales
**Se ejecuta UNA vez**, cuando todo el entrenamiento (02, 02b, 03) ha terminado y `protocolo.md` sigue congelado.
1. Verifica que cada checkpoint carga (antes de gastar tiempo).
2. Evalúa el test (`evaluar_test.py`): original, benchmark, baselines y ablaciones, hasta 10 semillas.
3. Estadística E3: original vs top 3 (T1–T3 + Friedman, Holm, δ de Cliff) y ablaciones vs completo.
4. Tablas del benchmark (`analisis_estadistico.py`).
5. ONNX + perfil; calibración, robustez y Grad-CAM para original, top 3 y U-Net.
**nnU-Net queda fuera de esta evaluación** (su pipeline 02b no terminó): `evaluar_test.py` y ONNX ya lo excluyen por defecto y `analisis_estadistico.py` lleva `--excluir nnunet`; debe declararse en el informe como limitación. Si más adelante termina `colab_02b`, se añade sin repetir lo demás. Corte de sesión: repite la celda; lo ya evaluado se salta.

In [ ]:
# Celda 1 — GPU
import os
!nvidia-smi | head -15

In [ ]:
# Celda 2 — dependencias
%pip install -q segmentation-models-pytorch monai einops timm scipy scikit-image pandas opencv-python-headless tqdm pytest matplotlib onnx onnxruntime
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")
assert torch.cuda.is_available(), "Sin GPU: activa una GPU (Runtime → Change runtime type)"

In [ ]:
# Celda 3 — montar Drive y fijar rutas
from google.colab import drive
drive.mount('/content/drive')
CARPETA_DRIVE = "/content/drive/MyDrive/proyecto_tb"      # AJUSTAR si cambia
RAIZ_LOCAL = "/content/proyecto_tb"                        # copia local (rápida)
RES_HPO = f"{CARPETA_DRIVE}/resultados"                    # donde quedó la búsqueda (hpo/)
RES_E3 = f"{CARPETA_DRIVE}/resultados_e3"                  # resultados finales del Entregable 3
HPO_CFG = f"{RES_HPO}/hpo/mejor_config.json"
SPL = f"{RAIZ_LOCAL}/datos/splits"
os.makedirs(RES_E3, exist_ok=True)
assert os.path.isdir(f"{CARPETA_DRIVE}/codigo"), "Falta codigo/ en Drive"
assert os.path.isdir(f"{CARPETA_DRIVE}/datos/splits"), "Falta datos/splits"
print("Drive OK:", CARPETA_DRIVE, "\nResultados E3:", RES_E3)

In [ ]:
# Celda 4 — copiar código y datos a disco local: train/val + TEST + GT (el test se lee solo aquí)
import csv, shutil
def a_local(r, raiz):
    n = r.replace("\\", "/"); i = n.find("datos/"); assert i >= 0, r
    return os.path.join(raiz, n[i:])
shutil.copytree(f"{CARPETA_DRIVE}/codigo", f"{RAIZ_LOCAL}/codigo", dirs_exist_ok=True)
os.makedirs(SPL, exist_ok=True)
for nombre in ("train", "val", "test_pos", "test_neg", "test_pos_reserva"):
    p = f"{CARPETA_DRIVE}/datos/splits/{nombre}.csv"
    if not os.path.exists(p): print("  (no existe", nombre, ")"); continue
    filas = list(csv.DictReader(open(p, newline="", encoding="utf-8")))
    campos = list(filas[0].keys())
    for f in filas:
        dst, src = a_local(f["ruta"], RAIZ_LOCAL), a_local(f["ruta"], CARPETA_DRIVE)
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy2(src, dst)
        f["ruta"] = dst
    with open(f"{SPL}/{nombre}.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=campos)
        w.writeheader(); w.writerows(filas)
    print(f"  {nombre:16s} {len(filas)} filas")
for mf in ("split_manifest_v2.json", "split_manifest.json"):
    if os.path.exists(f"{CARPETA_DRIVE}/datos/splits/{mf}"):
        shutil.copy2(f"{CARPETA_DRIVE}/datos/splits/{mf}", f"{SPL}/{mf}")

gt = f"{CARPETA_DRIVE}/datos/ground_truth_test"
assert os.path.isdir(gt), "Falta datos/ground_truth_test en Drive (GT plata)"
shutil.copytree(gt, f"{RAIZ_LOCAL}/datos/ground_truth_test", dirs_exist_ok=True)
print("GT plata copiado")
import sys; sys.path.insert(0, f"{RAIZ_LOCAL}/codigo")

In [ ]:
# Celda — congelar / verificar protocolo.md (SHA-256)
import hashlib
p = f"{CARPETA_DRIVE}/docs/protocolo.md"
assert os.path.exists(p), "Falta docs/protocolo.md en Drive"
h = hashlib.sha256(open(p, "rb").read()).hexdigest()
f = f"{RES_E3}/protocolo.sha256"
if os.path.exists(f):
    assert open(f).read().split()[0] == h, "protocolo.md CAMBIÓ después de congelarse: registra una enmienda"
    print("protocolo verificado:", h[:16], "…")
else:
    open(f, "w").write(h + "  protocolo.md\n"); print("protocolo CONGELADO:", h)

In [ ]:
# Celda 5 — control: ¿cargan todos los checkpoints? (no lee el test)
!python "{RAIZ_LOCAL}/codigo/evaluar_test.py" --resultados "{RES_E3}" --splits-dir "{SPL}" --verificar

In [ ]:
# Celda 6 — plan de la evaluación
!python "{RAIZ_LOCAL}/codigo/evaluar_test.py" --resultados "{RES_E3}" --splits-dir "{SPL}" --dry-run | tail -30

In [ ]:
# Celda 7 — EVALUAR EL TEST (una lectura por checkpoint congelado; reanudable)
!python "{RAIZ_LOCAL}/codigo/evaluar_test.py" --resultados "{RES_E3}" --splits-dir "{SPL}" --batch 4

In [ ]:
# Celda 8 — estadística E3: original vs top 3
!python "{RAIZ_LOCAL}/codigo/estadistica_e3.py" --resultados "{RES_E3}" --original lacca_unet

In [ ]:
# Celda 9 — ablaciones: cada variante contra el modelo completo (ΔDice > 0 = el componente ayuda)
!python "{RAIZ_LOCAL}/codigo/estadistica_e3.py" --resultados "{RES_E3}" --original lacca_unet --rivales lacca_eje_fijo lacca_sin_asimetria lacca_global --semillas 42 123 777 2024 31415

In [ ]:
# Celda 10 — tablas del benchmark (Friedman, post-hoc, bootstrap, AUC/DeLong)
!python "{RAIZ_LOCAL}/codigo/analisis_estadistico.py" --resultados "{RES_E3}" --splits-dir "{SPL}" --excluir nnunet

In [ ]:
# Celda 11 — ONNX + perfil (parámetros, FLOPs, latencia, memoria) de todos los modelos con checkpoint
!python "{RAIZ_LOCAL}/codigo/exportar_onnx.py" --resultados "{RES_E3}" --splits-dir "{SPL}"

In [ ]:
# Celda 12 — calibración, robustez y Grad-CAM: original, top 3 y U-Net (semilla 42)
import json
top = json.load(open(f"{RES_E3}/top3.json"))["top"]
for m in ["lacca_unet"] + top + ["unet"]:
    print("\n=====", m)
    !python "{RAIZ_LOCAL}/codigo/analisis_extra.py" --resultados "{RES_E3}" --modelo {m} --semilla 42 --splits-dir "{SPL}" --salida "{RES_E3}/extra"

In [ ]:
# Celda 13 — auditoría final de resultados
!python "{RAIZ_LOCAL}/codigo/limpiar_resultados.py" "{RES_E3}/resumen_runs.csv"
!cat "{RES_E3}/analisis_e3/resumen_e3.md"

In [ ]:
import shutil
shutil.move(f"{RES_E3}/analisis_e3", f"{RES_E3}/analisis_e3_ablaciones")
!python "{RAIZ_LOCAL}/codigo/estadistica_e3.py" --resultados "{RES_E3}"

## Después
Con esto quedan: tablas, comparaciones con Holm y δ de Cliff, ablaciones, calibración, robustez, Grad-CAM y ONNX. **Avísame con los resultados** para armar las conclusiones (incluida la validación externa de `colab_05_externos.ipynb`).